# Multimodal RAG with Gemini and Google Cloud

**Guided-learning project / portfolio adaptation.** Based on Google Cloud’s introductory multimodal RAG lab. This notebook demonstrates text/image retrieval and question answering over sample Alphabet financial reports. It is **not** a novel research contribution or a validated QA benchmark.

**Publishing note:** temporary cloud project IDs, generated outputs, and API credentials have been removed. You need your own authorized project and supported models to rerun API cells. See [README](../README.md) and [Attribution](../NOTICE.md).

In [ ]:
%pip install -q google-genai pymupdf pillow numpy pandas rich ipython


## 1. Configure the Google Gen AI SDK

This notebook uses a local compatibility helper for SDK differences encountered during the learning lab. Configure `GOOGLE_CLOUD_PROJECT` in your environment before running.

In [ ]:
import os
from google import genai
import lab_rag_compat as utils
from google.genai import types

# Use your own authorized Google Cloud project, not temporary lab credentials.
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT in your terminal and restart JupyterLab.")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "global")
CLIENT_MODE = os.environ.get("GENAI_CLIENT_MODE", "enterprise").strip().lower()

if CLIENT_MODE == "enterprise":
    client = genai.Client(enterprise=True, project=PROJECT_ID, location=LOCATION)
elif CLIENT_MODE == "vertexai":
    client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
else:
    raise ValueError("GENAI_CLIENT_MODE must be enterprise or vertexai")

utils.set_global_client(client)
print("Gen AI client initialized (mode: %s)." % CLIENT_MODE)


In [ ]:
from IPython.display import Markdown, display
from rich.markdown import Markdown as rich_Markdown
from google.genai import types
from PIL import Image as PIL_Image

In [ ]:
import os
model = os.environ.get("GENERATION_MODEL", "gemini-3.5-flash-lite")
print("Generation model:", model)


## 2. Download public sample documents

The original lab uses shortened Alphabet 10-K examples plus sample query images. This command retrieves them to your local workspace. Files are excluded from Git by default; confirm source terms before redistributing.

In [ ]:
!gcloud storage rsync -x ".*_utils\.py$" -r gs://github-repo/rag/intro_multimodal_rag/intro_multimodal_rag_old_version .
print("Data download command completed. Check the four source file paths below.")


In [ ]:
from pathlib import Path

for name in [
    "data/google-10k-sample-part1.pdf",
    "data/google-10k-sample-part2.pdf",
    "class_a_share.png",
    "tac_table_revenue.png",
]:
    print(name, Path(name).exists())

## 3. Extract and index multimodal metadata

Extract PDF text chunks and embedded figures, generate image descriptions, then cache matching text and image embeddings. The notebook caches intermediate data to avoid unnecessary API calls.

In [ ]:
from lab_rag_compat import get_document_metadata

In [ ]:
from pathlib import Path
import pandas as pd

pdf_folder_path = "data/"
image_description_prompt = """Explain what is going on in the image.
If it's a table, extract all elements of the table.
If it's a graph, explain the findings in the graph.
Do not include any numbers that are not mentioned in the image."""

cache_dir = Path('metadata')
text_path = cache_dir / 'text_metadata.pkl'
image_path = cache_dir / 'image_metadata.pkl'

if text_path.exists() and image_path.exists():
    print('Reusing previously extracted metadata; no API reprocessing required.')
    text_metadata_df = pd.read_pickle(text_path)
    image_metadata_df = pd.read_pickle(image_path)
else:
    text_metadata_df, image_metadata_df = get_document_metadata(
        model, pdf_folder_path, image_save_dir='images',
        image_description_prompt=image_description_prompt,
        embedding_size=1408,
        add_sleep_after_page=True, sleep_time_after_page=5,
    )
    cache_dir.mkdir(exist_ok=True)
    text_metadata_df.to_pickle(text_path)
    image_metadata_df.to_pickle(image_path)
print('Text rows:',len(text_metadata_df),'Image rows:',len(image_metadata_df))

In [ ]:
text_metadata_df.head()

In [ ]:
image_metadata_df.head()

In [ ]:
from lab_rag_compat import (
    display_images,
    get_gemini_response,
    get_similar_image_from_query,
    get_similar_text_from_query,
    print_text_to_image_citation,
    print_text_to_text_citation,
)

## 4. Text-based semantic search

Retrieve relevant chunks for a finance question and inspect citations. Retrieved page references are not proof that generated claims are supported; verify them manually.

In [ ]:
query = "I need details for basic and diluted net income per share of Class A, Class B, and Class C share for google?"

In [ ]:
# Matching user text query with "chunk_embedding" to find relevant chunks.
matching_results_text = get_similar_text_from_query(
    query,
    text_metadata_df,
    column_name="text_embedding_chunk",
    top_n=3,
    chunk_text=True,
)

# Print the matched text citations
print_text_to_text_citation(matching_results_text, print_top=False, chunk_text=True)

In [ ]:
print("\n **** Result: ***** \n")

# All relevant text chunk found across documents based on user query
context = "\n".join(
    [value["chunk_text"] for key, value in matching_results_text.items()]
)

instruction = f"""Answer the question with the given context.
If the information is not available in the context, just return "not available in the context".
Question: {query}
Context: {context}
Answer:
"""

# Prepare the model input
model_input = instruction

# Generate Gemini response with streaming output
get_gemini_response(
    model,  # we are passing Gemini
    model_input=model_input,
    stream=True,
    generation_config=types.GenerateContentConfig(temperature=0.2),
)

## 5. Text-to-image retrieval

Search visual descriptions, then inspect the retrieved figure.


In [ ]:
matching_results_image = get_similar_image_from_query(
    text_metadata_df,
    image_metadata_df,
    query=query,
    column_name="text_embedding_from_image_description",  # Use image description text embedding
    image_emb=False,  # Use text embedding instead of image embedding
    top_n=3,
    embedding_size=1408,
)

# Markdown(print_text_to_image_citation(matching_results_image, print_top=True))
print("\n **** Result: ***** \n")

# Display the top matching image
display(matching_results_image[0]["image_object"])

In [ ]:
print("\n **** Result: ***** \n")

context = f"""Image description: {matching_results_image[0]['image_description']}"""
instruction = f"""Answer the question in JSON format with the given image and its description.
Only include values supported by the image.
Question: {query}
Context: {context}
Answer:"""

print(get_gemini_response(
    model,
    model_input=[instruction, matching_results_image[0]['image_object']],
    stream=True,
    generation_config=types.GenerateContentConfig(temperature=1),
))

In [ ]:
print_text_to_image_citation(matching_results_image, print_top=True)


## 6. Image-to-image retrieval

Retrieve figures using the embedding of a query image rather than a text query.

In [ ]:
# You can find a similar image as per the images you have in the metadata.
# In this case, you have a table (picked from the same document source) and you would like to find similar tables in the document.
image_query_path = "tac_table_revenue.png"

# Print a message indicating the input image
print("***Input image from user:***")

# Display the input image
PIL_Image.open(image_query_path)

In [ ]:
# Search for Similar Images Based on Input Image and Image Embedding

matching_results_image = get_similar_image_from_query(
    text_metadata_df,
    image_metadata_df,
    query=query,  # Use query text for additional filtering (optional)
    column_name="mm_embedding_from_img_only",  # Use image embedding for similarity calculation
    image_emb=True,
    image_query_path=image_query_path,  # Use input image for similarity calculation
    top_n=3,  # Retrieve top 3 matching images
    embedding_size=1408,  # Use embedding size of 1408
)

print("\n **** Result: ***** \n")

# Display the Top Matching Image
display(
    matching_results_image[0]["image_object"]
)  # Display the top matching image object (Pillow Image)

In [ ]:
# Display citation details for the top matching image
print_text_to_image_citation(
    matching_results_image, print_top=True
)  # Print citation details for the top matching image

In [ ]:
# Check Other Matched Images (Optional)
# You can access the other two matched images using:

print("---------------Matched Images------------------\n")
display_images(
    [
        matching_results_image[0]["img_path"],
        matching_results_image[1]["img_path"],
    ],
    resize_ratio=0.5,
)

## 7. Comparative visual reasoning

Retrieve stock-performance plots and ask the multimodal model to compare them. The returned explanation is illustrative, not investment advice or an accuracy evaluation.

In [ ]:
matching_results_image_query_1 = get_similar_image_from_query(
    text_metadata_df,
    image_metadata_df,
    query="Show me all the graphs that shows Google Class A cumulative 5-year total return",
    column_name="text_embedding_from_image_description",  # Use image description text embedding # mm_embedding_from_img_only text_embedding_from_image_description
    image_emb=False,  # Use text embedding instead of image embedding
    top_n=3,
    embedding_size=1408,
)

In [ ]:
# Check Matched Images
# You can access the other two matched images using:

print("---------------Matched Images------------------\n")
display_images(
    [
        matching_results_image_query_1[0]["img_path"],
        matching_results_image_query_1[1]["img_path"],
    ],
    resize_ratio=0.5,
)

In [ ]:
prompt = f"""Compare the images and their extracted descriptions to answer these questions:
Image 1: {matching_results_image_query_1[0]['image_description']}
Image 2: {matching_results_image_query_1[1]['image_description']}
Questions:
- What are the key findings for Class A shares?
- How do the two graphs differ?
- How does Class A compare with the S&P 500?
- Which independent index most closely tracks Class A and why?
- Describe major patterns in both graphs.
Only use the evidence in the images."""

try:
    result = get_gemini_response(
        model,
        model_input=[prompt,
            matching_results_image_query_1[0]['image_object'],
            matching_results_image_query_1[1]['image_object']],
        stream=True,
        generation_config=types.GenerateContentConfig(temperature=1),
    )
    display(Markdown(result))
except Exception as exc:
    print('Comparison generation unavailable:', str(exc)[:220])
    print('The remaining notebook sections can still be inspected. This is not a successful comparison result.')

## 8. End-to-end multimodal RAG

Retrieve text and figures jointly, create a grounded prompt, request an answer, and print the supporting document references.

In [ ]:
# this time we are not passing any images, but just a simple text query.

query = """Questions:
 - What are the critical difference between various graphs for Class A Share?
 - Which index best matches Class A share performance closely where Google is not already a part? Explain the reasoning.
 - Identify key chart patterns for Google Class A shares.
 - What is cost of revenues, operating expenses and net income for 2020. Do mention the percentage change
 - What was the effect of Covid in the 2020 financial year?
 - What are the total revenues for APAC and USA for 2021?
 - What is deferred income taxes?
 - How do you compute net income per share?
 - What drove percentage change in the consolidated revenue and cost of revenue for the year 2021 and was there any effect of Covid?
 - What is the cause of 41% increase in revenue from 2020 to 2021 and how much is dollar change?
 """

In [ ]:
# Retrieve relevant chunks of text based on the query
matching_results_chunks_data = get_similar_text_from_query(
    query,
    text_metadata_df,
    column_name="text_embedding_chunk",
    top_n=10,
    chunk_text=True,
)

In [ ]:
# Get all relevant images based on user query
matching_results_image_fromdescription_data = get_similar_image_from_query(
    text_metadata_df,
    image_metadata_df,
    query=query,
    column_name="text_embedding_from_image_description",
    image_emb=False,
    top_n=10,
    embedding_size=1408,
)

In [ ]:
# combine all the selected relevant text chunks
context_text = []
for key, value in matching_results_chunks_data.items():
    context_text.append(value["chunk_text"])
final_context_text = "\n".join(context_text)

# combine all the relevant images and their description generated by Gemini
context_images = []
for key, value in matching_results_image_fromdescription_data.items():
    context_images.extend(
        ["Image: ", value["image_object"], "Caption: ", value["image_description"]]
    )

In [ ]:
prompt = f"""Use only the retrieved context to answer the financial questions.
If there is not enough evidence, respond 'Not enough context to answer'.
Text context:
{final_context_text[:10000]}
Image captions:
{chr(10).join(value['image_description'] for value in matching_results_image_fromdescription_data.values())[:4500]}
Questions:
{query}
Provide concise source-aware reasoning."""

try:
    result = get_gemini_response(
        model,
        model_input=[prompt] + [
           r['image_object'] for r in list(matching_results_image_fromdescription_data.values())[:2]
        ],
        stream=True,
        generation_config=types.GenerateContentConfig(temperature=1, max_output_tokens=1024),
    )
    display(Markdown(result))
except Exception as exc:
    print('RAG generation unavailable:', str(exc)[:220])
    print('Continue to the citation cells below; they identify retrieved sources but do not validate generated claims.')

In [ ]:
print('---------------Matched Images------------------\n')
paths = [r['img_path'] for r in list(matching_results_image_fromdescription_data.values())[:4]]
display_images(paths, resize_ratio=0.5)

In [ ]:
# Image citations. You can check how Gemini generated metadata helped in grounding the answer.

print_text_to_image_citation(
    matching_results_image_fromdescription_data, print_top=False
)

In [ ]:
# Text citations

print_text_to_text_citation(
    matching_results_chunks_data,
    print_top=False,
    chunk_text=True,
)